# Anomaly Detection — PyTorch Autoencoder

This notebook demonstrates:
- Feature engineering from order transactions
- Training an autoencoder on clean data
- Scoring all orders with reconstruction error
- Visualising the anomaly score distribution
- Evaluating against ground truth (injected anomalies)


In [ ]:
import sys
sys.path.insert(0, '..')
from dotenv import load_dotenv
load_dotenv('../.env')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch

from src.utils.spark_session import get_spark_session
from src.utils.paths import SILVER_PATH, MODEL_SAVE_PATH

spark = get_spark_session()
sns.set_theme(style='whitegrid', palette='husl')

In [ ]:
# ── Train model ───────────────────────────────────────────────────────────────
from src.models.train import main as train_main
train_main()

In [ ]:
# ── Run inference ─────────────────────────────────────────────────────────────
from src.models.inference import run_inference
n_anomalies, threshold = run_inference()
print(f'Anomalies detected: {n_anomalies}, Threshold: {threshold:.6f}')

In [ ]:
# ── Load scores ───────────────────────────────────────────────────────────────
scores_pdf = spark.read.format('delta').load(str(SILVER_PATH / 'anomaly_scores')).toPandas()
print(f'Total scored: {len(scores_pdf):,}')
scores_pdf.head()

In [ ]:
# ── Reconstruction error distribution ────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Distribution
axes[0].hist(scores_pdf['reconstruction_error'], bins=100, color='steelblue', alpha=0.7, label='All orders')
axes[0].axvline(threshold, color='red', linestyle='--', linewidth=2, label=f'Threshold ({threshold:.4f})')
axes[0].set_xlabel('Reconstruction Error')
axes[0].set_ylabel('Count')
axes[0].set_title('Anomaly Score Distribution')
axes[0].legend()

# Anomaly type breakdown
type_counts = scores_pdf['anomaly_type_predicted'].value_counts()
axes[1].bar(type_counts.index, type_counts.values, color=['steelblue' if t == 'normal' else 'tomato' for t in type_counts.index])
axes[1].set_xlabel('Anomaly Type')
axes[1].set_ylabel('Count')
axes[1].set_title('Orders by Anomaly Type')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig('../governance/reports/anomaly_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Top anomalous transactions ────────────────────────────────────────────────
top_anomalies = scores_pdf[scores_pdf['is_anomaly_predicted']].sort_values('reconstruction_error', ascending=False)
print(f'Top 10 most anomalous transactions:')
top_anomalies[['order_id', 'total_amount', 'reconstruction_error', 'anomaly_type_predicted']].head(10)

In [ ]:
# ── Precision/Recall against ground truth ─────────────────────────────────────
# Join with original orders to get ground truth
orders_pdf = spark.read.format('delta').load(str(SILVER_PATH / 'orders')).toPandas()
merged = scores_pdf.merge(orders_pdf[['order_id', 'is_anomaly_injected']], on='order_id', how='left')
merged['is_anomaly_injected'] = merged['is_anomaly_injected'].fillna(False)

from sklearn.metrics import classification_report, confusion_matrix

print('\nClassification Report:')
print(classification_report(merged['is_anomaly_injected'], merged['is_anomaly_predicted'],
                             target_names=['normal', 'anomaly']))

cm = confusion_matrix(merged['is_anomaly_injected'], merged['is_anomaly_predicted'])
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['pred_normal', 'pred_anomaly'],
            yticklabels=['true_normal', 'true_anomaly'])
plt.title('Confusion Matrix')
plt.show()